### Dataset Schema Overview

#### Core clinical tables

| Table              | Keys (PK / main FKs)                                      | Role      |
|--------------------|-----------------------------------------------------------|-----------|
| `patients.csv`     | PK: `Id`; used as `PATIENT` / `PATIENTID`                 | Primary   |
| `encounters.csv`   | PK: `Id`; FKs: `PATIENT` → patients, `ORGANIZATION` → organizations, `PROVIDER` → providers, `PAYER` → payers | Primary   |
| `observations.csv` | FKs: `PATIENT`, `ENCOUNTER`                               | Primary   |
| `conditions.csv`   | FKs: `PATIENT`, `ENCOUNTER`                               | Primary   |
| `procedures.csv`   | FKs: `PATIENT`, `ENCOUNTER`                               | Primary   |
| `medications.csv`  | FKs: `PATIENT`, `ENCOUNTER`, `PAYER`                      | Primary   |

#### Context / enrichment

| Table                 | Keys (PK / main FKs)                      | Role      |
|-----------------------|-------------------------------------------|-----------|
| `careplans.csv`       | PK: `Id`; FK: `PATIENT`, `ENCOUNTER`      | Secondary |
| `devices.csv`         | FK: `PATIENT`, `ENCOUNTER`                | Secondary |
| `imaging_studies.csv` | PK: `Id`; FK: `PATIENT`, `ENCOUNTER`      | Secondary |
| `allergies.csv`       | FK: `PATIENT`, `ENCOUNTER`                | Secondary |
| `immunizations.csv`   | FK: `PATIENT`, `ENCOUNTER`                | Secondary |
| `supplies.csv`        | FK: `PATIENT`, `ENCOUNTER`                | Sec/Ignore|
| `organizations.csv`   | PK: `Id`; used by `encounters`, `providers` | Secondary |
| `providers.csv`       | PK: `Id`; FK: `ORGANIZATION`              | Secondary |

#### Financial / coverage (usually ignore for label)

| Table                    | Keys (PK / main FKs)                   | Role   |
|--------------------------|----------------------------------------|--------|
| `payers.csv`             | PK: `Id`                               | Ignore |
| `payer_transitions.csv`  | FK: `PATIENT`, `PAYER`                 | Ignore |
| `claims.csv`             | PK: `Id`; FK: patient/provider/payer   | Ignore |
| `claims_transactions.csv`| PK: `ID`; FK: `CLAIMID`, `PATIENTID`   | Ignore |

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np


In [2]:
DATA_DIR = Path("../data/raw")

patients = pd.read_csv(DATA_DIR / "patients.csv")
encounters = pd.read_csv(DATA_DIR / "encounters.csv")
conditions = pd.read_csv(DATA_DIR / "conditions.csv")
observations = pd.read_csv(DATA_DIR / "observations.csv")
medications = pd.read_csv(DATA_DIR / "medications.csv")
procedures = pd.read_csv(DATA_DIR / "procedures.csv")

In [3]:
for table_name, df in zip(
    ['patients','encounters','observations','conditions','medications'],
    [patients, encounters, observations, conditions, medications]
):
    print(f"--- {table_name} ---")
    print(df.shape)
    print(df.dtypes)
    print(df.isna().sum().sort_values(ascending=False).head(10))
    print("\n")

--- patients ---
(1163, 25)
Id                         str
BIRTHDATE                  str
DEATHDATE                  str
SSN                        str
DRIVERS                    str
PASSPORT                   str
PREFIX                     str
FIRST                      str
LAST                       str
SUFFIX                     str
MAIDEN                     str
MARITAL                    str
RACE                       str
ETHNICITY                  str
GENDER                     str
BIRTHPLACE                 str
ADDRESS                    str
CITY                       str
STATE                      str
COUNTY                     str
ZIP                    float64
LAT                    float64
LON                    float64
HEALTHCARE_EXPENSES    float64
HEALTHCARE_COVERAGE    float64
dtype: object
SUFFIX       1147
DEATHDATE    1000
MAIDEN        832
ZIP           545
MARITAL       384
PASSPORT      276
PREFIX        245
DRIVERS       215
Id              0
ADDRESS         0
dty

In [21]:

# -------------------------------------------------------------------
# 1) Type normalization helpers
# -------------------------------------------------------------------

def normalize_datetime_columns(df, candidate_cols):
    """Convert specified columns to tz-naive pandas datetime (UTC→naive) if they exist."""
    for col in candidate_cols:
        if col in df.columns:
            dt = pd.to_datetime(df[col], errors="coerce", utc=True)
            # Drop timezone to make tz-naive
            df[col] = dt.dt.tz_convert(None)
    return df

def coerce_numeric_columns(df, candidate_cols):
    """Convert specified columns to float if they exist."""
    for col in candidate_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

# Known datetime / numeric candidates in this schema
datetime_candidates = ["START", "STOP", "DATE", "BIRTHDATE", "DEATHDATE"]
numeric_candidates = ["VALUE", "BASE_COST", "TOTALCOST", "PAYER_COVERAGE", "DISPENSES"]

# Normalize each table defensively
for df_name in ["patients", "encounters", "conditions", "observations", "medications"]:
    df = globals().get(df_name, None)
    if isinstance(df, pd.DataFrame) and not df.empty:
        df = normalize_datetime_columns(df, datetime_candidates)
        df = coerce_numeric_columns(df, numeric_candidates)
        globals()[df_name] = df  # write back

# -------------------------------------------------------------------
# 2) Patient cleanup: drop irrelevant columns, create IS_DECEASED
# -------------------------------------------------------------------

if isinstance(patients, pd.DataFrame) and not patients.empty:
    drop_cols = ["SUFFIX", "MAIDEN", "PASSPORT", "DRIVERS", "PREFIX", "ADDRESS"]
    patients = patients.drop(columns=[c for c in drop_cols if c in patients.columns], errors="ignore")

    if "DEATHDATE" in patients.columns:
        patients["IS_DECEASED"] = np.where(patients["DEATHDATE"].notna(), 1, 0)
    else:
        patients["IS_DECEASED"] = 0

# -------------------------------------------------------------------
# 3) Base encounter table + patient demographics / age
# -------------------------------------------------------------------

# Start from encounters (1 row per encounter)
encounter_features = encounters.copy()

# Join selected patient demographic columns
patient_demo_cols = ["Id", "BIRTHDATE", "GENDER", "RACE", "ETHNICITY", "IS_DECEASED"]
patient_demo_cols = [c for c in patient_demo_cols if c in patients.columns]

if "PATIENT" in encounter_features.columns and "Id" in patients.columns:
    patients_demo = patients[patient_demo_cols].rename(columns={"Id": "PATIENT_ID"})
    encounter_features = encounter_features.merge(
        patients_demo,
        left_on="PATIENT",
        right_on="PATIENT_ID",
        how="left",
    )

# Compute AGE_AT_ENCOUNTER (years) from encounter START and patient BIRTHDATE
if "START" in encounter_features.columns and "BIRTHDATE" in encounter_features.columns:
    age_delta = encounter_features["START"] - encounter_features["BIRTHDATE"]
    # Use total seconds to preserve partial years
    encounter_features["AGE_AT_ENCOUNTER"] = age_delta.dt.total_seconds() / (365.25 * 24 * 3600)
else:
    encounter_features["AGE_AT_ENCOUNTER"] = np.nan

# -------------------------------------------------------------------
# 4a) Observation aggregates: mean/min/max VALUE per ENCOUNTER for top 50 CODEs
# -------------------------------------------------------------------

obs_enc_agg = None

if isinstance(observations, pd.DataFrame) and not observations.empty:
    usable = observations.copy()

    # Keep numeric VALUE only
    if "VALUE" in usable.columns:
        usable["VALUE"] = pd.to_numeric(usable["VALUE"], errors="coerce")
        usable = usable[usable["VALUE"].notna()]
    else:
        usable = usable.iloc[0:0]  # no VALUE column → empty

    # Proceed only if required columns exist and data remain
    if not usable.empty and {"ENCOUNTER", "CODE", "VALUE"}.issubset(usable.columns):
        # Top 50 most frequent observation codes
        top_codes = usable["CODE"].value_counts().head(50).index
        usable = usable[usable["CODE"].isin(top_codes)]

        if not usable.empty:
            obs_agg = (
                usable
                .groupby(["ENCOUNTER", "CODE"])["VALUE"]
                .agg(["mean", "min", "max"])
                .reset_index()
            )

            # Pivot to wide format
            obs_pivot = obs_agg.set_index(["ENCOUNTER","CODE"])[["mean","min","max"]].unstack("CODE")
            # Flatten MultiIndex columns: (stat, code) → "stat_OBS_<code>"
            obs_pivot.columns = [
                f"{stat}_OBS_{code}" for stat, code in obs_pivot.columns.to_flat_index()
            ]
            obs_pivot = obs_pivot.reset_index().rename(columns={"ENCOUNTER": "ENCOUNTER_ID"})
            obs_enc_agg = obs_pivot

# Merge observation features into encounter_features
if obs_enc_agg is not None and not obs_enc_agg.empty and "Id" in encounter_features.columns:
    encounter_features = encounter_features.merge(
        obs_enc_agg,
        left_on="Id",
        right_on="ENCOUNTER_ID",
        how="left",
    )
    encounter_features.drop(columns=["ENCOUNTER_ID"], inplace=True, errors="ignore")

# -------------------------------------------------------------------
# 4b) Medication aggregates: MEDICATION_COUNT, TOTAL_MED_COST per ENCOUNTER
# -------------------------------------------------------------------

med_enc_agg = None

if isinstance(medications, pd.DataFrame) and not medications.empty:
    meds = medications.copy()

    if "ENCOUNTER" in meds.columns:
        # Identify cost columns vs coverage columns separately
        cost_cols = [c for c in meds.columns if "COST" in c.upper() and "COVERAGE" not in c.upper()]
        coverage_cols = [c for c in meds.columns if "COVERAGE" in c.upper()]

        # Ensure numeric
        for c in cost_cols + coverage_cols:
            meds[c] = pd.to_numeric(meds[c], errors="coerce")

        # Row-level totals (prefer TOTALCOST if present)
        if "TOTALCOST" in meds.columns:
            meds["ROW_MED_COST"] = meds["TOTALCOST"]
        elif ("BASE_COST" in meds.columns) and ("DISPENSES" in meds.columns):
            meds["ROW_MED_COST"] = meds["BASE_COST"] * meds["DISPENSES"]
        elif "BASE_COST" in meds.columns:
            meds["ROW_MED_COST"] = meds["BASE_COST"]
        else:
            meds["ROW_MED_COST"] = np.nan

        # Row-level payer coverage
        if "PAYER_COVERAGE" in meds.columns:
            meds["ROW_MED_COVERAGE"] = meds["PAYER_COVERAGE"]
        else:
            meds["ROW_MED_COVERAGE"] = np.nan

        med_group = meds.groupby("ENCOUNTER").agg(
            TOTAL_MED_COST=("ROW_MED_COST", "sum"),
            TOTAL_MED_PAYER_COVERAGE=("ROW_MED_COVERAGE", "sum"),
            MEDICATION_COUNT=("CODE", pd.Series.nunique) if "CODE" in meds.columns else ("ROW_MED_COST", "size"),
        ).reset_index()

        # Optional: out-of-pocket estimate
        med_group["TOTAL_MED_OOP_EST"] = (med_group["TOTAL_MED_COST"] - med_group["TOTAL_MED_PAYER_COVERAGE"]).clip(lower=0)
    
        med_enc_agg = med_group.rename(columns={"ENCOUNTER": "ENCOUNTER_ID"})

# Merge medication features
if med_enc_agg is not None and not med_enc_agg.empty and "Id" in encounter_features.columns:
    encounter_features = encounter_features.merge(
        med_enc_agg,
        left_on="Id",
        right_on="ENCOUNTER_ID",
        how="left",
    )
    encounter_features.drop(columns=["ENCOUNTER_ID"], inplace=True, errors="ignore")

# -------------------------------------------------------------------
# 4c) Condition aggregates: CONDITION_COUNT per ENCOUNTER
# -------------------------------------------------------------------

cond_enc_agg = None

if isinstance(conditions, pd.DataFrame) and not conditions.empty:
    cond = conditions.copy()
    cond["DESCRIPTION"] = cond["DESCRIPTION"].fillna("").astype(str)

    # identify sepsis-related condition rows
    is_sepsis_condition = cond["DESCRIPTION"].str.contains(r"\b(sepsis|septic)\b", case=False, regex=True, na=False)

    # compute count excluding sepsis-related conditions
    cond_non_sepsis = cond[~is_sepsis_condition]

    cond_enc_agg = (
        cond_non_sepsis.groupby("ENCOUNTER")["CODE"]
        .nunique()
        .reset_index()
        .rename(columns={"CODE": "CONDITION_COUNT", "ENCOUNTER": "ENCOUNTER_ID"})
    )

# Merge condition features
if cond_enc_agg is not None and not cond_enc_agg.empty and "Id" in encounter_features.columns:
    encounter_features = encounter_features.merge(
        cond_enc_agg,
        left_on="Id",
        right_on="ENCOUNTER_ID",
        how="left",
    )
    encounter_features.drop(columns=["ENCOUNTER_ID"], inplace=True, errors="ignore")

# -------------------------------------------------------------------
# 5) Sepsis label (encounter-level, no leakage)
# -------------------------------------------------------------------

sepsis_enc = None

if isinstance(conditions, pd.DataFrame) and not conditions.empty:
    cond = conditions.copy()
    if {"ENCOUNTER", "DESCRIPTION"}.issubset(cond.columns):
        desc = cond["DESCRIPTION"].fillna("").astype(str)
        sepsis_flag = desc.str.contains(r"\b(?:sepsis|septic)\b", case=False, regex=True)
        cond["SEPSIS_FLAG"] = sepsis_flag.astype(int)

        sepsis_enc = (
            cond.groupby("ENCOUNTER")["SEPSIS_FLAG"]
            .max()  # any sepsis/septic in that encounter
            .reset_index()
            .rename(columns={"ENCOUNTER": "ENCOUNTER_ID", "SEPSIS_FLAG": "SEPSIS_LABEL"})
        )

# Default label = 0; overwrite where we have evidence
if "SEPSIS_LABEL" not in encounter_features.columns:
    encounter_features["SEPSIS_LABEL"] = 0

if sepsis_enc is not None and not sepsis_enc.empty and "Id" in encounter_features.columns:
    encounter_features = encounter_features.merge(
        sepsis_enc,
        left_on="Id",
        right_on="ENCOUNTER_ID",
        how="left",
        suffixes=("", "_SEPSIS_TMP"),
    )
    # Consolidate labels: use encounter-specific label where present
    if "SEPSIS_LABEL_SEPSIS_TMP" in encounter_features.columns:
        encounter_features["SEPSIS_LABEL"] = encounter_features["SEPSIS_LABEL_SEPSIS_TMP"].fillna(
            encounter_features["SEPSIS_LABEL"]
        )
        encounter_features.drop(columns=["SEPSIS_LABEL_SEPSIS_TMP"], inplace=True, errors="ignore")
    encounter_features.drop(columns=["ENCOUNTER_ID"], inplace=True, errors="ignore")

# Ensure SEPSIS_LABEL is integer {0,1}
encounter_features["SEPSIS_LABEL"] = encounter_features["SEPSIS_LABEL"].fillna(0).astype(int)

# -------------------------------------------------------------------
# 6) Final output: shape, label prevalence, preview
# -------------------------------------------------------------------

print("encounter_features.shape:", encounter_features.shape)
print("SEPSIS_LABEL prevalence (mean):", encounter_features["SEPSIS_LABEL"].mean())

encounter_features.head()

encounter_features.shape: (61459, 169)
SEPSIS_LABEL prevalence (mean): 0.0007484664573130054


/var/folders/47/wml7fp2j44v1sn39wh0mzbjm0000gn/T/ipykernel_16858/1813281173.py:189: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  is_sepsis_condition = cond["DESCRIPTION"].str.contains(r"\b(sepsis|septic)\b", case=False, regex=True, na=False)


,Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,...,max_OBS_8462-4,max_OBS_8480-6,max_OBS_8867-4,max_OBS_9279-1,TOTAL_MED_COST,TOTAL_MED_PAYER_COVERAGE,MEDICATION_COUNT,TOTAL_MED_OOP_EST,CONDITION_COUNT,SEPSIS_LABEL
0,748f8357-6cc7-551d-f31a-32fa2cf84126,2019-02-17 05:07:38,2019-02-17 05:22:38,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),...,89.0,115.0,94.0,15.0,NaN,NaN,NaN,NaN,NaN,0
1,5a4735ae-423f-6563-28ab-b3d11b49b2d4,2019-03-24 05:07:38,2019-03-24 05:22:38,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),...,71.0,115.0,89.0,14.0,NaN,NaN,NaN,NaN,NaN,0
2,0bee1ce6-3e2c-5506-f71c-a7ba8f64a3d3,2019-05-26 05:07:38,2019-05-26 05:22:38,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),...,77.0,124.0,95.0,16.0,NaN,NaN,NaN,NaN,NaN,0
3,6e93bcf9-45a4-8528-0120-1c1eaa930faf,2019-07-28 05:07:38,2019-07-28 05:22:38,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),...,82.0,112.0,88.0,13.0,NaN,NaN,NaN,NaN,NaN,0
4,8b6787c3-4316-a0cb-899d-4746525c319f,2019-10-27 05:07:38,2019-10-27 05:22:38,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,f7ae497d-8dc6-3721-9402-43b621a4e7d2,82608ebb-037c-3cef-9d34-3736d69b29e8,7c4411ce-02f1-39b5-b9ec-dfbea9ad3c1a,wellness,410620009,Well child visit (procedure),...,74.0,116.0,98.0,15.0,NaN,NaN,NaN,NaN,NaN,0


I end up with about 61k encounters and ~170 features. Using a strict diagnosis-text label gives ~0.07% positives, which is expectedly low. This is a conservative first-pass phenotype; a more realistic sepsis definition would incorporate labs and organ dysfunction criteria. Structurally everything looks correct, encounter-level aggregation worked and feature joins look sane.

##### This is a first-pass label using diagnosis text. In practice I’d build a clinical phenotype using labs like lactate, vitals, infection codes, and possibly timing relative to ICU admission. This version is intentionally simple.

In [22]:
# Drop metadata columns
id_like = ["Id",  "PATIENT_ID", "PROVIDER", "ORGANIZATION", "PAYER"]
encounter_features= encounter_features.drop(columns=[c for c in id_like if c in encounter_features.columns])

In [23]:
encounter_features["TOTAL_MED_COST"].info()

<class 'pandas.Series'>
RangeIndex: 61459 entries, 0 to 61458
Series name: TOTAL_MED_COST
Non-Null Count  Dtype  
--------------  -----  
26871 non-null  float64
dtypes: float64(1)
memory usage: 480.3 KB


In [24]:
# Make MEDICATION_COUNT an integer, and fill missing with 0
for c in ["TOTAL_MED_COST", "TOTAL_MED_PAYER_COVERAGE", "TOTAL_MED_OOP_EST"]:
    if c in encounter_features.columns:
        encounter_features[c] = encounter_features[c].fillna(0.0).astype("float64")

if "MEDICATION_COUNT" in encounter_features.columns:
    encounter_features["MEDICATION_COUNT"] = encounter_features["MEDICATION_COUNT"].fillna(0).astype(int)

#### Modeling & Evaluation

In [31]:
# -------------------------------------------------------------------
# Encounter-level modeling: prepare X/y, handle missingness, train baseline model
# -------------------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupShuffleSplit


# -------------------------------------------------------------------
# 0) Choose source dataframe: prefer encounter_features if available
# -------------------------------------------------------------------
if "encounter_features" in globals() and isinstance(encounter_features, pd.DataFrame):
    df = encounter_features.copy()
else:
    df = encounter_features.copy()

# Basic defensive check
if "SEPSIS_LABEL" not in df.columns:
    raise ValueError("SEPSIS_LABEL column not found in encounter-level dataframe.")

# -------------------------------------------------------------------
# 1) Create X and y, drop label + identifier columns
# -------------------------------------------------------------------
# y = label
y = df["SEPSIS_LABEL"].astype(int)
# groups for patient-level split
if "PATIENT" not in df.columns:
    raise ValueError("PATIENT column missing; keep it for group splitting.")
groups = df["PATIENT"]


# Drop label and known identifier columns
id_cols = ["SEPSIS_LABEL", "Id", "PATIENT_ID", "PROVIDER", "ORGANIZATION", "PAYER", "CODE", "REASONCODE"]
id_cols_present = [c for c in id_cols if c in df.columns]
X = df.drop(columns=id_cols_present)

# -------------------------------------------------------------------
# 2) One-hot encode key categoricals (if present)
# -------------------------------------------------------------------
cat_cols = [c for c in ["GENDER", "RACE", "ETHNICITY"] if c in X.columns]
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Ensure only numeric features go into the model (drop any remaining non-numerics like dates/strings)
X = X.select_dtypes(include=[np.number])

# -------------------------------------------------------------------
# 3) Handle missingness
#   - Replace +/-inf with NaN
#   - For count/cost columns: fill NaN with 0
#   - For remaining numeric: median impute (fitted on train only)
#   - Add missingness indicators for lab columns (mean/min/max_OBS_*)
# -------------------------------------------------------------------

# Replace infinities with NaN
X = X.replace([np.inf, -np.inf], np.nan)

# Identify lab/vital columns from observation aggregates
lab_prefixes = ("mean_OBS_", "min_OBS_", "max_OBS_")
lab_cols = [c for c in X.columns if c.startswith(lab_prefixes)]

# Add missingness indicators for lab columns BEFORE imputation
# for col in lab_cols:
#     X[f"{col}__missing"] = X[col].isna().astype(int)
missing_df = X[lab_cols].isna().astype(int)
missing_df.columns = [f"{c}__missing" for c in missing_df.columns]
X = pd.concat([X, missing_df], axis=1)

# Count/cost columns to fill with zero (if they exist)
count_cost_cols = [
    "MEDICATION_COUNT",
    "TOTAL_MED_COST",
    "TOTAL_MED_PAYER_COVERAGE",
    "TOTAL_MED_OOP_EST",
    "CONDITION_COUNT",
]
count_cost_cols_present = [c for c in count_cost_cols if c in X.columns]
for col in count_cost_cols_present:
    X[col] = X[col].fillna(0.0)

# -------------------------------------------------------------------
# 4) Train/test split (stratified), then median imputation using TRAIN medians
# -------------------------------------------------------------------
# Patient-level grouping to prevent leakage
#groups = encounter_features["PATIENT"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

print("Unique patients - train:", groups.iloc[train_idx].nunique())
print("Unique patients - test :", groups.iloc[test_idx].nunique())
print("Train positives:", int(y_train.sum()), "out of", len(y_train))
print("Test positives :", int(y_test.sum()), "out of", len(y_test))


# Median impute remaining numeric columns using TRAIN medians
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
# Only impute columns that still have missing values in TRAIN and are not count/cost
median_impute_cols = [
    c
    for c in numeric_cols
    if c not in count_cost_cols_present and X_train[c].isna().any()
]

for col in median_impute_cols:
    median_val = X_train[col].median()
    X_train.loc[:, col] = X_train[col].fillna(median_val)
    X_test.loc[:, col] = X_test[col].fillna(median_val)

# Sanity check: no remaining NaNs for model inputs
assert not X_train.isna().any().any(), "NaNs remain in X_train after imputation."
assert not X_test.isna().any().any(), "NaNs remain in X_test after imputation."

# -------------------------------------------------------------------
# 5) Train a class_weight="balanced" LogisticRegression and evaluate
# -------------------------------------------------------------------
# clf = LogisticRegression(
#     penalty="l2",
#     solver="saga",
#     class_weight="balanced",
#     max_iter=2000,
#     C=0.5,
#     n_jobs=-1,
#     random_state=42,
# )
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

clf = make_pipeline(
    StandardScaler(with_mean=False),  # safe for sparse-ish matrices
    LogisticRegression(class_weight="balanced", max_iter=5000, C=0.5, random_state=42)
)

clf.fit(X_train, y_train)

# Predictions (probabilities for positive class)
y_train_proba = clf.predict_proba(X_train)[:, 1]
y_test_proba = clf.predict_proba(X_test)[:, 1]

# Metrics
roc_auc_train = roc_auc_score(y_train, y_train_proba)
roc_auc_test = roc_auc_score(y_test, y_test_proba)

pr_auc_train = average_precision_score(y_train, y_train_proba)
pr_auc_test = average_precision_score(y_test, y_test_proba)

print("Train positives:", int(y_train.sum()), "out of", len(y_train))
print("Test positives :", int(y_test.sum()), "out of", len(y_test))
print(f"ROC-AUC  (train): {roc_auc_train:.4f}")
print(f"ROC-AUC  (test) : {roc_auc_test:.4f}")
print(f"PR-AUC   (train): {pr_auc_train:.4f}")
print(f"PR-AUC   (test) : {pr_auc_test:.4f}")

# -------------------------------------------------------------------
# 6) Top 15 absolute coefficient features (sanity check)
# -------------------------------------------------------------------
lr = clf.named_steps["logisticregression"]

coef = lr.coef_.ravel()
feature_names = np.array(X_train.columns)

# Sort by absolute coefficient magnitude
top_idx = np.argsort(np.abs(coef))[::-1][:15]
top_features = feature_names[top_idx]
top_coefs = coef[top_idx]

print("\nTop 15 features by |coefficient|:")
for name, val in zip(top_features, top_coefs):
    print(f"{name:40s}  coef = {val:+.4f}")

Unique patients - train: 930
Unique patients - test : 233
Train positives: 38 out of 50465
Test positives : 8 out of 10994
Train positives: 38 out of 50465
Test positives : 8 out of 10994
ROC-AUC  (train): 0.9979
ROC-AUC  (test) : 0.7339
PR-AUC   (train): 0.3429
PR-AUC   (test) : 0.1047

Top 15 features by |coefficient|:
CONDITION_COUNT                           coef = -5.6160
BASE_ENCOUNTER_COST                       coef = -5.0205
TOTAL_MED_PAYER_COVERAGE                  coef = -4.9740
MEDICATION_COUNT                          coef = +2.4888
min_OBS_8480-6                            coef = -1.8369
mean_OBS_8480-6                           coef = -1.6301
AGE_AT_ENCOUNTER                          coef = -1.6227
mean_OBS_8867-4                           coef = -1.6066
max_OBS_8480-6                            coef = -1.5378
max_OBS_9279-1                            coef = -1.1977
IS_DECEASED                               coef = +0.8802
max_OBS_1975-2                            coef = +

### Solid first pass (encounter-level baseline)

**Class imbalance**
- Train: 38 positives / 50K (~0.075%)
- Test: 8 positives / 11K (~0.073%)

**Performance**
- ROC-AUC: moderate generalization gap (expected with extreme imbalance + linear model + many features)
- Test PR-AUC: **0.1047**

Baseline PR-AUC ≈ prevalence ≈ 0.0007  
→ **~150× better than random**


### Top signals
- MEDICATION_COUNT  
- TOTAL_MED_PAYER_COVERAGE  
- CONDITION_COUNT  
- AGE_AT_ENCOUNTER  
- IS_DECEASED  
- Vitals (BP, SpO₂, HR)

These are clinically plausible:
- Older patients show higher sepsis risk  
- Medication count acts as a proxy for acuity  
- Hypotension / abnormal vitals appear strongly  

### Notable observation

Condition count appears inversely correlated, likely because wellness encounters have many coded diagnoses while sepsis visits are sparse acute admissions.

This likely reflects dataset structure rather than true clinical causality.

Next steps:

- Remove post-event financial features (costs / coverage)
- Restrict features to early encounter window (e.g., first 6–12h)
- Add temporal deltas on vitals/labs
- Try tree-based model (HistGB / XGBoost)
- Evaluate recall at fixed precision